In [1]:
#%pip install subprocess
import uproot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import subprocess
from enum import IntEnum, auto
from pathlib import Path

In [2]:
class Setting(IntEnum):
    g_nReal = auto()
    g_nZ = auto()
    g_nAMass = auto()
    g_nConEBin = auto()
    alpha_parameter = auto()
    sigma_parameter = auto()
    fine_binning = auto()
    rough_binning = auto()
    E_steps = auto()

setting_definer = {
    Setting.g_nReal : "const int g_nReal = ",
    Setting.g_nZ : "const int g_nZ = ",
    Setting.g_nAMass : "const int g_nAMass = ",
    Setting.g_nConEBin : "const int g_nConEBin = "
}

value_setting = {Setting.g_nReal,
                 Setting.g_nZ,
                 Setting.g_nAMass,
                 Setting.g_nConEBin}

In [29]:
def get_nld(spin_array, s):
    return spin_array[s]

def get_smooth(nld):
    pd_data = pd.Series(nld)
    fine = pd_data.rolling(window=settings[Setting.fine_binning], center=True).mean()
    rough = pd_data.rolling(window=settings[Setting.rough_binning], center=True).mean()
    return fine, rough

def analysis(s, energy, spin_array, *, print_nld = True, print_smooth = True, print_stationary = True, print_autocorrelation = True, print_comparison = True):
    fa_folder = Path(r"C:\RAINIER\sample_folder\fluctuation_analysis")
    file_name = (str(s)+".png")
    nld = get_nld(spin_array, s)

    if print_nld:
        plt.figure()
        for k in range(20):
            pass
            #plt.plot(energy, spin_array[k])
        plt.plot(energy, nld)
        plt.xlabel("E in MeV")
        plt.ylabel("Levels in Energy Bin")
        plt.title("NLD für alle spins")
        plt.yscale("log")
        plt.savefig(fa_folder / "NLD_input" / file_name, dpi=300)
        plt.close()

    fine, rough = get_smooth(nld)
    if print_smooth:
        plt.figure()
        plt.plot(energy, nld)
        plt.plot(energy, fine)
        plt.plot(energy, rough)
        plt.xlabel("E in MeV")
        plt.ylabel("Levels in Energy Bin")
        plt.title("Rough/fine smoothing")
        plt.yscale("log")
        plt.savefig(fa_folder / "Smoothing" / file_name, dpi=300)
        plt.close()

    d_full = fine/rough
    if print_stationary:
        plt.figure()
        plt.plot(energy, fine/rough)
        plt.savefig(fa_folder / "Stationary" / file_name, dpi=300)
        plt.close()


    def get_interval(myData, lower, upper):
        intervalled_data = []
        for k in range(myData.size):
            if energy[k] >= lower and energy[k] <= upper:
                intervalled_data.append(myData[k])
        intervalled_data = np.array(intervalled_data)
        return intervalled_data




    def auto(x, full_data = d_full, lower = -1, upper = -2):
        h1 = get_interval(full_data, lower, upper)
        h2 = get_interval(full_data, lower+x, upper+x)
        c1 = 0
        c2 = 0
        while h1.size > h2.size:
            c1 += 1
            h1 = h1[:-1]
        while h1.size < h2.size:
            c2 += 1
            h2 = h2[:-1]
        return np.mean(h1*h2)/(np.mean(h1)*np.mean(h2))

    eps_steps = 0.001
    epsilons = np.arange(0.0,0.5,eps_steps)
    if print_autocorrelation:
        plt.figure()
        plt.plot(epsilons, [auto(eps) for eps in epsilons])
        plt.xlim(0,0.5) 
        plt.savefig(fa_folder / "Autocorrelation" / file_name, dpi=300)
        plt.close()

    def lvl_dens(E_start, E_int_length, sigma = settings[Setting.sigma_parameter], alpha = settings[Setting.alpha_parameter]):
        return 1/ ((auto(0, d_full, E_start, E_start + E_int_length)-1)*2*sigma*np.sqrt(np.pi)/alpha)


    E_steps = settings[Setting.E_steps]
    final_E = np.arange(0,7, E_steps)
    if print_comparison:
        plt.figure()
        plt.scatter(final_E, [lvl_dens(E, E_steps) for E in final_E], color = "orange")
        plt.plot(energy, nld)
        plt.yscale("log")
        plt.title("Fluctuation Analysis compared to original level scheme")
        plt.xlabel("E in MeV")
        plt.ylabel("Levels per MeV")
        plt.savefig(fa_folder / "Comparison" / file_name, dpi=300)
        plt.close()

def run_simulation():
    apply_settings()
    return subprocess.run(["cmd", "/c", "root", r"C:\RAINIER\RAINIER.C"], capture_output=True, text=True, cwd=r"C:\RAINIER\sample_folder").stdout

def get_level_data(run):
    cut = run.find("More levels exist at higher spins")
    #print(run[cut:])
    cut2 = run[cut:].find("E(MeV)")
    first = run[cut+cut2+10:]
    cut3 = first.find("Total Number of Levels")
    second = first[:cut3]
    return second

def get_level_arrays(text):
    energy = []
    spin_array = []
    for k in range(len(text)):
        ch = text[k]
        if ch != '.':
            continue
        a = 1
        while text[k-(a+1)].isdigit():
            a = a+1
        num = float(text[k-a:k+4])
        energy.append(num)
        l = 20
        b = 8
        spin_array_one_energy = [[] for x in range(20)]
        while l > 0:
            if text[k+b] == " ":
                b = b+1
            c = 1
            #print("b:",b)
            while text[k+b+c] != "|":
                c = c+1
            #print("c:",c)
            spin_val = text[k+b:k+b+c]
            #print(spin_val)
            spin_val = spin_val.replace("\n", "")
            spin_array_one_energy[l-1] = int(spin_val)
            #print("spin"+str(20-l)+":"+spin_val)
            #print("Spin"+str(19-l)+":"+text[k+b:k+b+c+2])
            b = b+c+1
            l = l-1
        #print(spin_array_one_energy)
        spin_array.append(spin_array_one_energy)
    energy = np.array(energy)
    spin_array = np.array(spin_array)
    spin_array = spin_array.T
    return (energy,spin_array)

def fluctuation_analysis(run, *, print_nld = True, print_smooth = True, print_stationary = True, print_autocorrelation = True, print_comparison = True):
    energy, spin_array = get_level_arrays(get_level_data(run))
    fa_folder =  Path(r"C:\RAINIER\sample_folder\fluctuation_analysis")
    fa_folder.mkdir(exist_ok=True)
    (fa_folder/"NLD_input").mkdir(exist_ok=True)
    (fa_folder/"Smoothing").mkdir(exist_ok=True)
    (fa_folder/"Stationary").mkdir(exist_ok=True)
    (fa_folder/"Autocorrelation").mkdir(exist_ok=True)
    (fa_folder/"Comparison").mkdir(exist_ok=True)
    for s in range(spin_array.shape[0]):
        analysis(s, energy, spin_array, print_nld = print_nld, print_smooth = print_smooth, print_stationary = print_stationary, print_autocorrelation = print_autocorrelation, print_comparison = print_comparison)

def replace_val(text, definer, new_val):
    pos = text.find(definer)
    start_val = pos + len(definer)
    end_val = start_val
    while text[end_val] != ";":
        end_val = end_val + 1
    val = text[start_val:end_val]
    if val != str(new_val):
        print("(changed) "+text[pos:start_val] + str(new_val))
        return text[:start_val] + str(new_val) + text[end_val:]
    else:
        print(text[pos:end_val])
        return text

def apply_settings():
    with open(r"C:\RAINIER\sample_folder\settings.h", "r") as f:
        text = f.read()
    with open(r"C:\RAINIER\sample_folder\settings.h", "w") as f:
        for key in settings.keys():
            if key in value_setting:
                text = replace_val(text, setting_definer[key], settings[key])
        f.write(text)

In [28]:
settings = {
    Setting.g_nReal : 1,
    Setting.g_nZ : 26,
    Setting.g_nAMass : 56,
    Setting.g_nConEBin : 600,
    Setting.alpha_parameter : 0.45,
    Setting.sigma_parameter : 5,
    Setting.fine_binning : 10,
    Setting.rough_binning : 16,
    Setting.E_steps : 0.2
}

#run = run_simulation()
fluctuation_analysis(run, print_nld=False, print_smooth=False, print_stationary=False, print_autocorrelation=False, print_comparison=True)